In [ ]:
!pip -q install pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Libraries loaded!")

Libraries loaded!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving archive.zip to archive.zip


In [ ]:
import zipfile
import os

zip_name = list(uploaded.keys())[0]

extract_path = "/content/twitter_data"

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")

Dataset extracted!


In [ ]:
csv_path = "/content/twitter_data/twcs/twcs.csv"

df = pd.read_csv(
    csv_path,
    dtype={
        "tweet_id": str,
        "author_id": str,
        "response_tweet_id": str,
        "in_response_to_tweet_id": str
    }
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2811774, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6


In [ ]:
tweet_lookup = df.set_index("tweet_id")

support_replies = df[
    (df["author_id"] == "AppleSupport") &
    (df["inbound"] == False) &
    (df["in_response_to_tweet_id"].notna())
].copy()

support_replies["customer_text"] = support_replies[
    "in_response_to_tweet_id"
].map(tweet_lookup["text"])

support_replies["customer_id"] = support_replies[
    "in_response_to_tweet_id"
].map(tweet_lookup["author_id"])

support_pairs = support_replies[
    support_replies["customer_text"].notna()
].copy()

print("AppleSupport conversations:", len(support_pairs))

AppleSupport conversations: 106648


In [ ]:
support_pairs["customer_text"] = (
    support_pairs["customer_text"]
    .fillna("")
    .astype(str)
)

support_pairs["clean_text"] = (
    support_pairs["customer_text"]
    .str.lower()
    .str.replace(r"@\w+", " ", regex=True)
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

display(
    support_pairs[
        ["customer_text", "clean_text"]
    ].head()
)

,customer_text,clean_text
396,@AppleSupport https://t.co/NV0yucs0lB,
398,@AppleSupport The newest update. I️ made sure ...,the newest update. i️ made sure to download it...
401,@AppleSupport Tried resetting my settings .. r...,tried resetting my settings .. restarting my p...
403,@AppleSupport This is what it looks like https...,this is what it looks like
405,@AppleSupport I️ have an iPhone 7 Plus and yes...,i️ have an iphone 7 plus and yes i️ do


In [ ]:
INTENTS = [
    "Software / iOS Update",
    "Device / Hardware Problem",
    "Battery / Charging",
    "App Problem",
    "Apple Services",
    "Connectivity",
    "Messaging / Notifications",
    "Account / Apple ID",
    "Purchase / Activation",
    "Payment / Billing",
    "Complaint / Feedback",
    "Other / Unclear"
]

print("Number of intents:", len(INTENTS))

for i, intent in enumerate(INTENTS, 1):
    print(i, "-", intent)

Number of intents: 12
1 - Software / iOS Update
2 - Device / Hardware Problem
3 - Battery / Charging
4 - App Problem
5 - Apple Services
6 - Connectivity
7 - Messaging / Notifications
8 - Account / Apple ID
9 - Purchase / Activation
10 - Payment / Billing
11 - Complaint / Feedback
12 - Other / Unclear


In [ ]:
uploaded_golden = files.upload()

Saving golden_set_200_labeled.csv to golden_set_200_labeled.csv


In [ ]:
golden_file = list(uploaded_golden.keys())[0]

golden = pd.read_csv(golden_file)

print("Golden set:", golden.shape)
print(golden["intent"].value_counts())

Golden set: (200, 4)
intent
Software / iOS Update        58
Other / Unclear              47
Device / Hardware Problem    25
Battery / Charging           14
Connectivity                 12
App Problem                   9
Complaint / Feedback          9
Messaging / Notifications     9
Apple Services                7
Purchase / Activation         6
Account / Apple ID            3
Payment / Billing             1
Name: count, dtype: int64


In [ ]:
payment_rows = golden[
    golden["intent"] == "Payment / Billing"
].copy()

other_rows = golden[
    golden["intent"] != "Payment / Billing"
].copy()

train_golden, test = train_test_split(
    other_rows,
    test_size=40,
    random_state=42,
    stratify=other_rows["intent"]
)

# Payment/Billing has only one example,
# so keep it away from the test set.
train_golden = pd.concat(
    [train_golden, payment_rows],
    ignore_index=True
)

print("Golden training examples:", len(train_golden))
print("Golden test examples:", len(test))

Golden training examples: 160
Golden test examples: 40


In [ ]:
def assign_intent(text):
    text = str(text).lower()

    # Battery / Charging
    if re.search(
        r"\bbattery\b|\bcharging\b|\bcharger\b|\bbattery health\b|\bbattery drain\b|\bdrain(ed|ing)?\b",
        text
    ):
        return "Battery / Charging"

    # Messaging / Notifications
    if re.search(
        r"\bimessage\b|\bmessages?\b|\bsms\b|\bnotification(s)?\b|\btext message\b",
        text
    ):
        return "Messaging / Notifications"

    # Account / Apple ID
    if re.search(
        r"\bapple id\b|\bforgot password\b|\bpassword\b|\blogin\b|\blog in\b|\bsign in\b|\bsigning in\b",
        text
    ):
        return "Account / Apple ID"

    # Payment / Billing
    if re.search(
        r"\bpayment\b|\bbilling\b|\bbilled\b|\brefund\b|\bcharged for\b|\bcredit card\b|\bdebit card\b",
        text
    ):
        return "Payment / Billing"

    # Purchase / Activation
    if re.search(
        r"\bactivation\b|\bactivate\b|\bactivated\b|\bpurchase\b|\bpurchased\b|\border\b|\bbuy\b",
        text
    ):
        return "Purchase / Activation"

    # Connectivity
    if re.search(
        r"\bwi[- ]?fi\b|\bbluetooth\b|\binternet\b|\bnetwork\b|\bconnection\b|\bconnected\b|\bconnect\b",
        text
    ):
        return "Connectivity"

    # Apple Services
    if re.search(
        r"\bicloud\b|\bapple music\b|\bitunes\b|\bfind my\b|\bfacetime\b|\bsiri\b|\bpodcast\b",
        text
    ):
        return "Apple Services"

    # Software / iOS Update
    if re.search(
        r"\bios\b.*\bupdate\b|\bupdate\b.*\bios\b|\bios\s*\d|\bmacos\b|\bsoftware update\b|\bsoftware\b.*\bupdate\b",
        text
    ):
        return "Software / iOS Update"

    # App Problem
    if re.search(
        r"\bapp store\b|\bapps?\b.*\b(crash|crashing|broken|work|working|open|opening|download)\b|\b(crash|crashing)\b.*\bapps?\b",
        text
    ):
        return "App Problem"

    # Device / Hardware
    if re.search(
        r"\bscreen\b|\bcamera\b|\bspeaker\b|\bkeyboard\b|\btrackpad\b|\biphone\b.*\b(broken|freeze|freezing)\b|\bipad\b.*\b(broken|freeze|freezing)\b|\bmac\b.*\b(broken|freeze|freezing)\b|\bfinder\b",
        text
    ):
        return "Device / Hardware Problem"

    # Complaint / Feedback
    if re.search(
        r"\bawful\b|\bterrible\b|\bworst\b|\bhorrible\b|\bdisappointed\b|\bpiece of shit\b|\bfix your\b",
        text
    ):
        return "Complaint / Feedback"

    # Everything else
    return "Other / Unclear"

In [ ]:
support_pairs["intent"] = support_pairs["clean_text"].apply(
    assign_intent
)

print(
    support_pairs["intent"].value_counts()
)

intent
Other / Unclear              61462
Software / iOS Update        12049
Battery / Charging            9247
Device / Hardware Problem     4638
Apple Services                4219
Messaging / Notifications     3960
Connectivity                  3922
Purchase / Activation         2290
Account / Apple ID            1505
Complaint / Feedback          1483
App Problem                   1205
Payment / Billing              668
Name: count, dtype: int64


In [ ]:
golden_ids = set(
    golden["tweet_id"].astype(str)
)

historical_train = support_pairs[
    ~support_pairs["tweet_id"].astype(str).isin(golden_ids)
].copy()

print("Historical data available:", len(historical_train))

Historical data available: 106448


In [ ]:
train_100k = historical_train.sample(
    n=min(100000, len(historical_train)),
    random_state=42
).copy()

print("Training on:", len(train_100k))
print("\nIntent distribution:")
print(train_100k["intent"].value_counts())

NameError: name 'historical_train' is not defined

In [ ]:
X_train = train_100k["customer_text"].fillna("")
y_train = train_100k["intent"]

model_100k = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

model_100k.fit(
    X_train,
    y_train
)

print("MODEL TRAINED ON 100K DATA!")

MODEL TRAINED ON 100K DATA!


In [ ]:
X_test = test["customer_text"].fillna("")
y_test = test["intent"]

y_pred = model_100k.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("================================")
print("100K TRAINING RESULTS")
print("================================")

print("Accuracy :", round(accuracy, 4))
print("Macro F1 :", round(macro_f1, 4))

100K TRAINING RESULTS
Accuracy : 0.475
Macro F1 : 0.4753


In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

                           precision    recall  f1-score   support

       Account / Apple ID       1.00      1.00      1.00         1
              App Problem       1.00      0.50      0.67         2
           Apple Services       0.00      0.00      0.00         1
       Battery / Charging       1.00      1.00      1.00         3
     Complaint / Feedback       0.00      0.00      0.00         2
             Connectivity       1.00      0.50      0.67         2
Device / Hardware Problem       1.00      0.20      0.33         5
Messaging / Notifications       1.00      0.50      0.67         2
          Other / Unclear       0.29      0.78      0.42         9
    Purchase / Activation       0.00      0.00      0.00         1
    Software / iOS Update       0.80      0.33      0.47        12

                 accuracy                           0.47        40
                macro avg       0.64      0.44      0.48        40
             weighted avg       0.68      0.47      0.48    

In [ ]:
results = pd.DataFrame({
    "customer_message": test["customer_text"].values,
    "true_intent": y_test.values,
    "predicted_intent": y_pred
})

display(results)

,customer_message,true_intent,predicted_intent
0,@AppleSupport Any way to check the overall bat...,Battery / Charging,Battery / Charging
1,@AppleSupport the new Mac OS Sierra just messe...,Device / Hardware Problem,Other / Unclear
2,hey @AppleSupport My find my iphone alert just...,Other / Unclear,Apple Services
3,@AppleSupport same result https://t.co/RLmsqNKfkX,Other / Unclear,Other / Unclear
4,@115858 did the iOS 11.03 update and my batter...,Battery / Charging,Battery / Charging
5,"@115858 , this started happening after the upd...",Software / iOS Update,Other / Unclear
6,"@AppleSupport Hi , i have an iPhone 8+ and i h...",Device / Hardware Problem,Other / Unclear
7,@AppleSupport Thanks for the reply. I’m runnin...,Software / iOS Update,Other / Unclear
8,@AppleSupport IOS 11.2,Software / iOS Update,Software / iOS Update
9,@AppleSupport Can y’all fix that problem,Other / Unclear,Other / Unclear


In [ ]:
wrong = results[
    results["true_intent"] != results["predicted_intent"]
]

print("Wrong predictions:", len(wrong))

display(wrong)

Wrong predictions: 21


,customer_message,true_intent,predicted_intent
1,@AppleSupport the new Mac OS Sierra just messe...,Device / Hardware Problem,Other / Unclear
2,hey @AppleSupport My find my iphone alert just...,Other / Unclear,Apple Services
5,"@115858 , this started happening after the upd...",Software / iOS Update,Other / Unclear
6,"@AppleSupport Hi , i have an iPhone 8+ and i h...",Device / Hardware Problem,Other / Unclear
7,@AppleSupport Thanks for the reply. I’m runnin...,Software / iOS Update,Other / Unclear
10,@115858 @AppleSupport who can I chat/ contact ...,Purchase / Activation,Other / Unclear
11,@AppleSupport this new update has completely J...,Software / iOS Update,Other / Unclear
12,@AppleSupport why do the alarms on my iPhone 6...,Device / Hardware Problem,Software / iOS Update
14,Anyone have any idea why my @115858 TV downloa...,Apple Services,Other / Unclear
15,@115858 your new update messed up my phone😭 th...,Software / iOS Update,Other / Unclear


Model 2

In [ ]:
from sklearn.svm import LinearSVC

svm_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced"
        )
    )
])

svm_model.fit(
    train_100k["customer_text"],
    train_100k["intent"]
)

print("Linear SVM trained on 100K data!")

Linear SVM trained on 100K data!


In [ ]:
svm_pred = svm_model.predict(
    test["customer_text"]
)

svm_accuracy = accuracy_score(
    test["intent"],
    svm_pred
)

svm_f1 = f1_score(
    test["intent"],
    svm_pred,
    average="macro"
)

print("================================")
print("LINEAR SVM RESULTS")
print("================================")
print("Accuracy :", round(svm_accuracy, 4))
print("Macro F1 :", round(svm_f1, 4))

LINEAR SVM RESULTS
Accuracy : 0.5
Macro F1 : 0.4981


In [ ]:
print(
    classification_report(
        test["intent"],
        svm_pred,
        zero_division=0
    )
)

                           precision    recall  f1-score   support

       Account / Apple ID       1.00      1.00      1.00         1
              App Problem       1.00      0.50      0.67         2
           Apple Services       0.00      0.00      0.00         1
       Battery / Charging       1.00      1.00      1.00         3
     Complaint / Feedback       0.00      0.00      0.00         2
             Connectivity       1.00      0.50      0.67         2
Device / Hardware Problem       1.00      0.40      0.57         5
Messaging / Notifications       1.00      0.50      0.67         2
          Other / Unclear       0.30      0.78      0.44         9
    Purchase / Activation       0.00      0.00      0.00         1
    Software / iOS Update       0.80      0.33      0.47        12

                 accuracy                           0.50        40
                macro avg       0.65      0.46      0.50        40
             weighted avg       0.68      0.50      0.51    

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy
    ],
    "Macro F1": [
        macro_f1,
        svm_f1
    ]
})

display(comparison)

,Model,Accuracy,Macro F1
0,Logistic Regression,0.475,0.475288
1,Linear SVM,0.500,0.498138


Model 3


In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        MultinomialNB()
    )
])

nb_model.fit(
    train_100k["customer_text"],
    train_100k["intent"]
)

print("Multinomial Naive Bayes trained on 100K data!")

Multinomial Naive Bayes trained on 100K data!


In [ ]:
nb_pred = nb_model.predict(
    test["customer_text"]
)

nb_accuracy = accuracy_score(
    test["intent"],
    nb_pred
)

nb_f1 = f1_score(
    test["intent"],
    nb_pred,
    average="macro"
)

print("================================")
print("NAIVE BAYES RESULTS")
print("================================")
print("Accuracy :", round(nb_accuracy, 4))
print("Macro F1 :", round(nb_f1, 4))

NAIVE BAYES RESULTS
Accuracy : 0.35
Macro F1 : 0.1541


In [ ]:
print(
    classification_report(
        test["intent"],
        nb_pred,
        zero_division=0
    )
)

                           precision    recall  f1-score   support

       Account / Apple ID       0.00      0.00      0.00         1
              App Problem       0.00      0.00      0.00         2
           Apple Services       0.00      0.00      0.00         1
       Battery / Charging       1.00      1.00      1.00         3
     Complaint / Feedback       0.00      0.00      0.00         2
             Connectivity       0.00      0.00      0.00         2
Device / Hardware Problem       0.00      0.00      0.00         5
Messaging / Notifications       0.00      0.00      0.00         2
          Other / Unclear       0.26      1.00      0.41         9
    Purchase / Activation       0.00      0.00      0.00         1
    Software / iOS Update       1.00      0.17      0.29        12

                 accuracy                           0.35        40
                macro avg       0.21      0.20      0.15        40
             weighted avg       0.43      0.35      0.25    

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Multinomial Naive Bayes"
    ],
    "Training Data": [
        "100K",
        "100K",
        "100K"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy,
        nb_accuracy
    ],
    "Macro F1": [
        macro_f1,
        svm_f1,
        nb_f1
    ]
})

display(comparison.sort_values(
    "Macro F1",
    ascending=False
))

,Model,Training Data,Accuracy,Macro F1
1,Linear SVM,100K,0.500,0.498138
0,Logistic Regression,100K,0.475,0.475288
2,Multinomial Naive Bayes,100K,0.350,0.154073


In [ ]:
from sklearn.linear_model import RidgeClassifier

ridge_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        RidgeClassifier(
            class_weight="balanced"
        )
    )
])

ridge_model.fit(
    train_100k["customer_text"],
    train_100k["intent"]
)

print("Ridge Classifier trained on 100K data!")

Ridge Classifier trained on 100K data!


In [ ]:
ridge_pred = ridge_model.predict(
    test["customer_text"]
)

ridge_accuracy = accuracy_score(
    test["intent"],
    ridge_pred
)

ridge_f1 = f1_score(
    test["intent"],
    ridge_pred,
    average="macro"
)

print("================================")
print("RIDGE CLASSIFIER RESULTS")
print("================================")
print("Accuracy :", round(ridge_accuracy, 4))
print("Macro F1 :", round(ridge_f1, 4))

RIDGE CLASSIFIER RESULTS
Accuracy : 0.5
Macro F1 : 0.4812


In [ ]:
print(
    classification_report(
        test["intent"],
        ridge_pred,
        zero_division=0
    )
)

                           precision    recall  f1-score   support

       Account / Apple ID       1.00      1.00      1.00         1
              App Problem       1.00      0.50      0.67         2
           Apple Services       0.00      0.00      0.00         1
       Battery / Charging       1.00      0.67      0.80         3
     Complaint / Feedback       0.00      0.00      0.00         2
             Connectivity       1.00      0.50      0.67         2
Device / Hardware Problem       0.67      0.40      0.50         5
Messaging / Notifications       1.00      0.50      0.67         2
          Other / Unclear       0.33      0.78      0.47         9
    Purchase / Activation       0.00      0.00      0.00         1
    Software / iOS Update       0.71      0.42      0.53        12

                 accuracy                           0.50        40
                macro avg       0.61      0.43      0.48        40
             weighted avg       0.62      0.50      0.51    

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Multinomial Naive Bayes",
        "Ridge Classifier"
    ],
    "Training Data": [
        "100K",
        "100K",
        "100K",
        "100K"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy,
        nb_accuracy,
        ridge_accuracy
    ],
    "Macro F1": [
        macro_f1,
        svm_f1,
        nb_f1,
        ridge_f1
    ]
})

display(
    comparison.sort_values(
        "Macro F1",
        ascending=False
    )
)

,Model,Training Data,Accuracy,Macro F1
1,Linear SVM,100K,0.500,0.498138
3,Ridge Classifier,100K,0.500,0.481180
0,Logistic Regression,100K,0.475,0.475288
2,Multinomial Naive Bayes,100K,0.350,0.154073


Model 4

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=3,
            max_features=30000
        )
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=50,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])

rf_model.fit(
    train_100k["customer_text"],
    train_100k["intent"]
)

print("Random Forest trained!")

Random Forest trained!


In [ ]:
         rf_pred = rf_model.predict(
    test["customer_text"]
)

rf_accuracy = accuracy_score(
    test["intent"],
    rf_pred
)

rf_f1 = f1_score(
    test["intent"],
    rf_pred,
    average="macro"
)

print("================================")
print("RANDOM FOREST RESULTS")
print("================================")
print("Accuracy :", round(rf_accuracy, 4))
print("Macro F1 :", round(rf_f1, 4))

RANDOM FOREST RESULTS
Accuracy : 0.5
Macro F1 : 0.4795


In [ ]:
print(
    classification_report(
        test["intent"],
        rf_pred,
        zero_division=0
    )
)

                           precision    recall  f1-score   support

       Account / Apple ID       1.00      1.00      1.00         1
              App Problem       1.00      0.50      0.67         2
           Apple Services       0.00      0.00      0.00         1
       Battery / Charging       1.00      1.00      1.00         3
     Complaint / Feedback       0.00      0.00      0.00         2
             Connectivity       1.00      0.50      0.67         2
Device / Hardware Problem       1.00      0.20      0.33         5
Messaging / Notifications       1.00      0.50      0.67         2
          Other / Unclear       0.32      0.89      0.47         9
    Purchase / Activation       0.00      0.00      0.00         1
    Software / iOS Update       0.80      0.33      0.47        12

                 accuracy                           0.50        40
                macro avg       0.65      0.45      0.48        40
             weighted avg       0.69      0.50      0.49    

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Multinomial Naive Bayes",
        "Ridge Classifier",
        "Random Forest"
    ],
    "Training Data": [
        "100K",
        "100K",
        "100K",
        "100K",
        "100K"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy,
        nb_accuracy,
        ridge_accuracy,
        rf_accuracy
    ],
    "Macro F1": [
        macro_f1,
        svm_f1,
        nb_f1,
        ridge_f1,
        rf_f1
    ]
})

display(
    comparison.sort_values(
        "Macro F1",
        ascending=False
    )
)

,Model,Training Data,Accuracy,Macro F1
1,Linear SVM,100K,0.500,0.498138
3,Ridge Classifier,100K,0.500,0.481180
4,Random Forest,100K,0.500,0.479501
0,Logistic Regression,100K,0.475,0.475288
2,Multinomial Naive Bayes,100K,0.350,0.154073


In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        SGDClassifier(
            loss="log_loss",
            max_iter=1000,
            random_state=42,
            class_weight="balanced"
        )
    )
])

sgd_model.fit(
    train_100k["customer_text"],
    train_100k["intent"]
)

print("SGD Classifier trained on 100K data!")

SGD Classifier trained on 100K data!


In [ ]:
sgd_pred = sgd_model.predict(
    test["customer_text"]
)

sgd_accuracy = accuracy_score(
    test["intent"],
    sgd_pred
)

sgd_f1 = f1_score(
    test["intent"],
    sgd_pred,
    average="macro"
)

print("================================")
print("SGD CLASSIFIER RESULTS")
print("================================")
print("Accuracy :", round(sgd_accuracy, 4))
print("Macro F1 :", round(sgd_f1, 4))

SGD CLASSIFIER RESULTS
Accuracy : 0.5
Macro F1 : 0.4795


In [ ]:
print(
    classification_report(
        test["intent"],
        sgd_pred,
        zero_division=0
    )
)

                           precision    recall  f1-score   support

       Account / Apple ID       1.00      1.00      1.00         1
              App Problem       1.00      0.50      0.67         2
           Apple Services       0.00      0.00      0.00         1
       Battery / Charging       1.00      1.00      1.00         3
     Complaint / Feedback       0.00      0.00      0.00         2
             Connectivity       1.00      0.50      0.67         2
Device / Hardware Problem       1.00      0.20      0.33         5
Messaging / Notifications       1.00      0.50      0.67         2
          Other / Unclear       0.32      0.89      0.47         9
    Purchase / Activation       0.00      0.00      0.00         1
    Software / iOS Update       0.80      0.33      0.47        12

                 accuracy                           0.50        40
                macro avg       0.65      0.45      0.48        40
             weighted avg       0.69      0.50      0.49    

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Multinomial Naive Bayes",
        "Ridge Classifier",
        "SGD Classifier"
    ],
    "Training Data": [
        "100K",
        "100K",
        "100K",
        "100K",
        "100K"
    ],
    "Accuracy": [
        accuracy,
        svm_accuracy,
        nb_accuracy,
        ridge_accuracy,
        sgd_accuracy
    ],
    "Macro F1": [
        macro_f1,
        svm_f1,
        nb_f1,
        ridge_f1,
        sgd_f1
    ]
})

display(
    comparison.sort_values(
        "Macro F1",
        ascending=False
    )
)

,Model,Training Data,Accuracy,Macro F1
1,Linear SVM,100K,0.500,0.498138
3,Ridge Classifier,100K,0.500,0.481180
4,SGD Classifier,100K,0.500,0.479501
0,Logistic Regression,100K,0.475,0.475288
2,Multinomial Naive Bayes,100K,0.350,0.154073
